# 08. Visualization

1. **W&B:** training curves, configs, held-out predictions for every run.
2. **Grad-CAM (Faster R-CNN):** which backbone stage / FPN level explains a detection.
3. **RoI embeddings (t-SNE):** are classes and errors separated?
4. **RT-DETR attention:** where the decoder samples, by layer and object size.

All plots: fold-0 model on held-out fold-0 frames.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval wandb

In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.manifold import TSNE

from core import CLASSES, N_FOLDS, SEED
from core.cv import best_per_family, environment, existing
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.errors import match_detections
from core.tv import load_model
from core.variants import run_config
from core.viz import (attention_stats, cam_concentration, frcnn_gradcam, frcnn_roi_embeddings, overlay,
                      rtdetr_probe, wandb_log_run)
from core.yolo import load_ultralytics

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data: held-out fold 0

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
held0 = frames[frames["fold"] == 0]

# every ground-truth object of fold 0 with its COCO size bin
BINS = ["small (<32 px)", "medium", "large (>96 px)"]
objs = pd.DataFrame([{"key": r.key, "uid": r.uid, "x1": b[0], "y1": b[1], "x2": b[2], "y2": b[3], "cls": int(c)}
                     for r in held0.itertuples() for b, c in zip(*targets[r.uid][:2])])
objs["side"] = np.sqrt((objs.x2 - objs.x1) * (objs.y2 - objs.y1))
objs["bin"] = pd.cut(objs["side"] ** 2, [0, 32 ** 2, 96 ** 2, np.inf], labels=BINS)
path_of = dict(zip(held0["key"], held0["path"]))
rng = np.random.default_rng(SEED)


def load_rgb(key):
    return cv2.cvtColor(cv2.imread(path_of[key]), cv2.COLOR_BGR2RGB)


def crop_box(box, shape, scale=4, min_side=200):
    """Square window around a box, for zoomed-in figures."""
    H, W = shape[:2]
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    s = max(min_side, scale * max(box[2] - box[0], box[3] - box[1]))
    x0, y0 = int(np.clip(cx - s / 2, 0, max(W - s, 0))), int(np.clip(cy - s / 2, 0, max(H - s, 0)))
    return x0, y0, int(min(W, x0 + s)), int(min(H, y0 + s))


objs["bin"].value_counts()

## 1. Weights & Biases
Needs the Kaggle secret `WANDB_API_KEY`.

In [ ]:
PROJECT = "visdrone-vid-hw2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    pass  # outside Kaggle: use `wandb login` or the WANDB_API_KEY environment variable

RUNS = sorted({f.name[len("metrics_"):-len("_fold0.json")] for root in (WORK, Path("/kaggle/input")) if root.exists()
               for f in root.rglob("metrics_*_fold0.json")})
if os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_MODE") == "offline":
    urls = {name: wandb_log_run(name, WORK, frames, targets, PROJECT, N_FOLDS) for name in RUNS if existing(f"cv_{name}.json", WORK)}
    display(pd.Series(urls, name="W&B run"))
else:
    print("no W&B key: add the Kaggle secret WANDB_API_KEY to log", RUNS)

## 2. Grad-CAM on Faster R-CNN
Target: score of the detection covering the object. Only the FPN level the box is pooled from gets a gradient.

### Small vs large object, same frame

In [ ]:
FRCNN = best_per_family(["frcnn"], WORK)["frcnn"]
cfg = run_config(FRCNN, WORK)
model = load_model(existing(f"{FRCNN}_fold0.pt", WORK), cfg)
print("model:", FRCNN)


def cam_row(axes, obj, title):
    img = load_rgb(obj.key)
    box = (obj.x1, obj.y1, obj.x2, obj.y2)
    cams, info = frcnn_gradcam(model, img, box, obj.cls, cfg)
    x0, y0, x1, y1 = crop_box(box, img.shape)
    shown = img.copy()
    cv2.rectangle(shown, (int(obj.x1), int(obj.y1)), (int(obj.x2), int(obj.y2)), (0, 255, 0), 2)
    axes[0].imshow(shown[y0:y1, x0:x1])
    axes[0].set_title(f"{title}: {CLASSES[obj.cls]} {obj.side:.0f} px" + (f", score {info['score']:.2f}" if info else ", no detection"),
                      fontsize=8)
    maps = {k: v for k, v in (cams or {}).items() if v.max() > 0}  # the unused FPN levels are all zero
    for ax, (name, cam) in zip(axes[1:], maps.items()):
        ax.imshow(overlay(img, cam)[y0:y1, x0:x1])
        ax.set_title(name, fontsize=8)
    for ax in axes:
        ax.axis("off")


key = objs.groupby("key").size().idxmax()  # the densest frame
same = objs[(objs.key == key) & (objs.cls == objs[objs.key == key].cls.mode()[0])].sort_values("side")
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
cam_row(axes[0], same.iloc[len(same) // 10], "small")
cam_row(axes[1], same.iloc[-1], "large")
plt.tight_layout()
plt.savefig(WORK / "viz_gradcam_small_large.png", dpi=110)

### Small object: found vs missed

In [ ]:
oof0 = pd.read_parquet(existing(f"oof_{FRCNN}_fold0.parquet", WORK))
outcomes, det_outcomes = match_detections(oof0, held0, gt, score_thr=0.3)
small = outcomes[np.sqrt(outcomes.w * outcomes.h) < 32]
pick = {"detected": small[small.outcome == "detected"], "missed": small[small.outcome == "missed"]}
fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for row, (title, cand) in zip(axes, pick.items()):
    if cand.empty:
        print("no small object that is", title)
        continue
    o = cand.sample(1, random_state=SEED).iloc[0]
    obj = pd.Series({"key": o.key, "x1": o.x, "y1": o.y, "x2": o.x + o.w, "y2": o.y + o.h, "cls": int(o.cls),
                     "side": np.sqrt(o.w * o.h)})
    cam_row(row, obj, title)
plt.tight_layout()
plt.savefig(WORK / "viz_gradcam_found_missed.png", dpi=110)

### Which stage explains which object size?
30 objects per size bin. Concentration: 1 = uniform map, higher = focused on the object.

In [ ]:
rows = []
sample = pd.concat([objs[objs["bin"] == b].sample(min(30, (objs["bin"] == b).sum()), random_state=SEED) for b in BINS])
for o in sample.itertuples():
    cams, info = frcnn_gradcam(model, load_rgb(o.key), (o.x1, o.y1, o.x2, o.y2), o.cls, cfg)
    if cams is None:
        continue
    for name, cam in cams.items():
        rows.append({"bin": o.bin, "map": name, "used": cam.max() > 0,
                     "concentration": cam_concentration(cam, (o.x1, o.y1, o.x2, o.y2))})
cam_table = pd.DataFrame(rows)
fpn = cam_table[cam_table["map"].str.startswith("P") & cam_table["used"]]
display(pd.crosstab(fpn["bin"], fpn["map"], normalize="index").round(2).rename_axis("FPN level the box is pooled from", axis=1))
cam_table[cam_table["used"]].pivot_table(index="bin", columns="map", values="concentration", aggfunc="median",
                                         observed=True).round(1)

### Findings

## 3. RoI embeddings (t-SNE)
Box-head features of detections (score ≥ 0.3) in 40 frames. t-SNE keeps neighbourhoods, not distances.

In [ ]:
keys = rng.choice(det_outcomes["key"].unique(), min(40, det_outcomes["key"].nunique()), replace=False)
emb_rows, emb = [], []
for k in keys:
    d = det_outcomes[det_outcomes.key == k]
    boxes = np.column_stack([d.x, d.y, d.x + d.w, d.y + d.h])
    emb.append(frcnn_roi_embeddings(model, load_rgb(k), boxes, cfg))
    emb_rows.append(d[["key", "cls", "score", "outcome"]].assign(side=np.sqrt(d.w * d.h)))
emb, emb_rows = np.concatenate(emb), pd.concat(emb_rows, ignore_index=True)
z = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(emb)
emb_rows[["tx", "ty"]] = z
emb_rows["bin"] = pd.cut(emb_rows["side"] ** 2, [0, 32 ** 2, 96 ** 2, np.inf], labels=BINS)
del model
torch.cuda.empty_cache()

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (col, title) in zip(axes, [("cls", "predicted class"), ("bin", "object size"), ("outcome", "outcome")]):
    for value, g in emb_rows.groupby(col, observed=True):
        label = CLASSES[int(value)] if col == "cls" else value
        ax.scatter(g.tx, g.ty, s=6, label=label, alpha=0.7)
    ax.set_title(title)
    ax.legend(fontsize=7, markerscale=2)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig(WORK / "viz_roi_tsne.png", dpi=110)
emb_rows["outcome"].value_counts()

### Findings

## 4. RT-DETR deformable attention
Confident queries (score ≥ 0.5) in 40 frames: distance of sampling points from the box centre and attention
share per feature level (level 0 = highest resolution).

In [ ]:
RT = best_per_family(["rtdetr"], WORK)["rtdetr"]
rt_cfg = run_config(RT, WORK)
net = load_ultralytics(existing(f"{RT}_fold0.pt", WORK)).model.to("cuda" if torch.cuda.is_available() else "cpu")
print("model:", RT)

stats, probes = [], {}
for k in rng.choice(held0["key"], min(40, len(held0)), replace=False):
    p = rtdetr_probe(net, load_rgb(k), rt_cfg["imgsz"])
    conf = np.where(p["score"] >= 0.5)[0]
    if len(conf):
        stats.append(attention_stats(p, conf).assign(key=k))
        probes[k] = p
att = pd.concat(stats, ignore_index=True)
att["bin"] = pd.cut(att["size"] ** 2, [0, 32 ** 2, 96 ** 2, np.inf], labels=BINS)
levels = [c for c in att.columns if c.startswith("level")]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
att.groupby(["layer", "bin"], observed=True)["distance_rel"].median().unstack().plot(marker="o", ax=axes[0])
axes[0].set(xlabel="decoder layer", ylabel="distance / box size", title="Sampling points: distance from the box centre")
first_last = att[att.layer.isin([1, att.layer.max()])].groupby(["bin", "layer"], observed=True)[levels].mean()
first_last.plot.bar(stacked=True, ax=axes[1], rot=30)
axes[1].set(xlabel="", ylabel="share of attention weight", title="Attention per feature level, first vs last layer")
plt.tight_layout()
plt.savefig(WORK / "viz_rtdetr_attention.png", dpi=110)
first_last.round(2)

### Sampling points: small vs large query (first, middle, last layer)

In [ ]:
k, p = next(iter(probes.items()))
img = load_rgb(k)
conf = np.where(p["score"] >= 0.5)[0]
size = np.sqrt((p["boxes"][:, 2] - p["boxes"][:, 0]) * (p["boxes"][:, 3] - p["boxes"][:, 1]))
queries = {"small": conf[np.argmin(size[conf])], "large": conf[np.argmax(size[conf])]}
show = [0, len(p["layers"]) // 2, len(p["layers"]) - 1]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (title, q) in zip(axes, queries.items()):
    b = p["boxes"][q]
    x0, y0, x1, y1 = crop_box(b, img.shape, scale=6, min_side=300)
    ax.imshow(img[y0:y1, x0:x1])
    ax.add_patch(plt.Rectangle((b[0] - x0, b[1] - y0), b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=2))
    for li, colour in zip(show, ["gold", "orange", "red"]):
        xy, w = p["layers"][li]["xy"][q].reshape(-1, 2), p["layers"][li]["w"][q].reshape(-1)
        ax.scatter(xy[:, 0] - x0, xy[:, 1] - y0, s=600 * w, c=colour, alpha=0.6, label=f"layer {li + 1}")
    ax.set_xlim(0, x1 - x0)
    ax.set_ylim(y1 - y0, 0)
    ax.axis("off")
    ax.set_title(f"{title}: {CLASSES[int(p['cls'][q])]} {size[q]:.0f} px, score {p['score'][q]:.2f}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(WORK / "viz_rtdetr_points.png", dpi=110)

### Findings